# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets
#%reload_ext dotenv

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import WebBaseLoader
URL_Alex_Ross="https://www.newyorker.com/magazine/2024/04/22/what-is-noise"
loader = WebBaseLoader(URL_Alex_Ross, requests_kwargs={'verify':False})

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
docs = loader.load()

print(docs[0].page_content)
print(docs[0].metadata)

/Users/VanyaG1/DOC/PhD-Local/DSI_Cert_Local/Deploying_AI/deploying-ai/deploying-ai-env/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.newyorker.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th AnniversaryOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave this storySave this story“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanings run the gamut from the negative to the positive, from the overpowering to the mysterious, from anarchy to sublimity. The negative seems to lie at the root: etymologists trace the word to “nuisance” and “nausea.” Noise is what drives us mad; it sends the Grinch over the edge at Christmastime. (“Oh, the Noise! Noise! Noise! Noise!”) Noise is the sound of madness itself, the din with

In [4]:
#more readable format for the content and metadata of the document.
from IPython.display import display, Markdown

#display(Markdown(docs[0].page_content)) #works but too long to display in the notebook 
display(Markdown(f"**Metadata:** {docs[0].metadata}")) #another way of displaying the metadata

**Metadata:** {'source': 'https://www.newyorker.com/magazine/2024/04/22/what-is-noise', 'title': 'What Is Noise? | The New Yorker', 'description': 'Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it, Alex Ross writes.', 'language': 'en-US'}

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
#load OpenAI client with the API Gateway URL and the API key from the environment variable
import os
from openai import OpenAI
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='API_GATEWAY_KEY',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})



In [ ]:
from pydantic import BaseModel


class ArticleAnalysis(BaseModel):
    title: str
    author: str
    relevance: str
    summary: str 
    input_tokens: int
    output_tokens: int


system_prompt="You speak Victorian English."
#main prompt: include the content of the article and the instructions for the AI to follow to analyze the article.
prompt = f"""
    I will provide you with an article, do the following:
    
    1. Identify the article's title and author.
    2. Provide a statement on what is the relevance of the article to an AI professional in their professional development. Max 1 paragraph.
    3. Summarize the article in no more than 1000 tokens.
        
    The article is the following: 
    <article>
    {docs[0].page_content}
    </article>

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Relevance: <relevance>
    Summary: <summary>
"""

response = client.responses.parse(
    model="gpt-4o", #use a larger model for the summary generation, than for the evaluation
    input=[
        { 'role': 'system','content': system_prompt},
        { 'role': 'user','content': prompt,}], 
    text_format= ArticleAnalysis,
    temperature=0.5
)

display(Markdown(response.output_text))
summary = response.output_parsed.summary


{"title":"What Is Noise?","author":"Alex Ross","relevance":"For an AI professional, this article offers a rich exploration of the concept of noise, which is crucial in fields like data science and machine learning. Understanding noise as both a physical phenomenon and a metaphor for informational overload can enhance an AI professional's ability to manage data quality, optimize algorithms, and improve signal processing. The historical and cultural perspectives on noise also provide insights into human interaction with data, which can inform the development of more intuitive AI systems.","summary":"The article \"What Is Noise?\" by Alex Ross explores the multifaceted concept of noise, examining its historical, cultural, and scientific dimensions. Noise, often seen negatively, has roots in words like \"nuisance\" and \"nausea.\" However, it can also be majestic, as seen in religious texts and literature. The article delves into how different languages and cultures perceive noise, with terms like French \"bruit\" and German \"Lärm.\" Ross discusses personal experiences with noise, reflecting on how control over noise affects our perception of it, a sentiment echoed by Garret Keizer’s book on noise ethics.\n\nRoss traces the evolution of noise in music, highlighting avant-garde composers like John Cage and György Ligeti, who embraced noise as an artistic element. The article also touches on the societal implications of noise, noting how it exposes social divides and serves as a tool of power. Historical efforts to control noise, from the Industrial Revolution to modern urban settings, demonstrate the challenges in defining and regulating noise.\n\nThe narrative shifts to informational noise, tracing its rise alongside technological advancements. Ross discusses how noise intersects with information theory, referencing Claude Shannon’s work on signal processing. The article concludes by reflecting on noise as a form of resistance and liberation in art, questioning whether noise can truly be separated from music. Through this exploration, Ross emphasizes the complexity of noise as both a disruptive and enriching force in human life."  , "input_tokens":10929,"output_tokens":336}

In [ ]:
#response.model_dump() #use this to see the structure of the response object and access specific attributes (like input/output tokens)
#summary = response.output_parsed.summary
#print(summary)

{'id': 'resp_0dafc0bb89b753c300698ea1dd4a308195ac0300b4f1d8914c',
 'created_at': 1770955229.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4o-mini-2024-07-18',
 'object': 'response',
 'output': [{'id': 'msg_0dafc0bb89b753c300698ea1df4ee881959bf6f9973836a2e9',
   'content': [{'annotations': [],
     'text': '{"title":"What Is Noise?","author":"Alex Ross","relevance":"The article delineates the multifaceted concept of noise, which has profound implications for AI professionals, especially in understanding the complexities of data interpretation and the significance of filtering and signal processing in artificial intelligence systems. As AI technology increasingly engages with vast datasets, the distinction between noise and valuable information becomes critically relevant for effective decision-making and algorithmic development.","summary":"In \'What Is Noise?\', Alex Ross explores the concept of noise from various cultural, histo

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [26]:
#Define the summary test with 5 bespoke questions

#load deepeval
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric

from deepeval.models import GPTModel
MODEL= GPTModel(
    model='gpt-4o-mini', #use different model for evaluation from the one that was used to generate the summary to avoid bias in the evaluation
    temperature=0.1, #lower temperature for evaluation to get more deterministic responses
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    #api_key='any_value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}

)

#define the input: the web page that was loaded
input=docs[0].page_content
#define the actual output: the summary generated by the model
actual_output_first=response.output_text

#bespoke test questions
Q1="Does the author talk about their own experience with noise?"  #correct answer: yes
Q2="Does noise refer only to an acoustic phenomenon?" #correct answer: no
Q3="Are notifications and algorithmic suggestions considered noise?" #correct answer: yes
Q4="Is making noise associated with power and status?" #correct answer: yes
Q5="Is all noise bad?" #correct answer: no
Q6="In English, does the word 'noise' have a negative connotation?" #correct answer: yes

#first test: summarization metric using 5 yes/no questions
test_case = LLMTestCase(input=input, actual_output=summary)
summary_metric = SummarizationMetric(
    threshold=0.5,
    model=MODEL,
    assessment_questions=[Q1,Q2,Q3,Q4,Q5,Q6],
    verbose_mode=True
)


In [27]:
#define the evaluation for clarity, tone, bias
# 
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams


#Evaluate clarity of the response using GEval

clarity = GEval(
    name="Clarity",
    model=MODEL,
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)


#evaluate tone
professionalism = GEval(
    name="Professionalism",  #alternatively: empathy, directness
    model=MODEL,
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)


#define bias test procedure
bias = GEval(
    name="Bias", #alternatively: "PII Leakage", "Ethical Alignment","Diversity"
    model=MODEL,
    evaluation_steps=[
        "Check whether the output includes any evidence of bias, such as stereotypes or unfair generalizations.",
        "Identify any segments where the sentiment of the output is not aligned with the sentiment of the input.",
        "Identify any omitted information that is relevant to the overall meaning and sentiment of the input."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)



In [29]:
#Run all the evaluations

#evaluate all four metrics together
results=evaluate(test_cases=[test_case], metrics=[summary_metric,clarity, professionalism, bias])




print('------ RESULTS ------')
for r in results.test_results:
    for m in r.metrics_data:
        print("----Metric:", m.name)
        print("Score:", m.score)
        print("Passed:", m.success)
        print("Reason:", m.reason)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Bias [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

...

Truths (limit=None):
[
    "Noise has various meanings, ranging from negative to positive.",
    "The word 'noise' is etymologically linked to 'nuisance' and 'nausea.'",
    "Noise can drive people mad, as illustrated by the Grinch's reaction to it.",
    "The Psalms contain references to joyful noise.",
    "In the Book of Ezekiel, God's voice is described as 'like a noise of many waters.'",
    "Public Enemy's song 'Bring the Noise' is an example of noise being used in a cultural context.",
    "In Elizabethan England, 'noyse' referred to a musical ensemble.",
    "Noise has been studied in various cultural histories and philosophies.",
    "Samuel Johnson stated that music is the least disagreeable of all noises.",
    "The distinction between noise and music is often subjective and personal.",
    "Noise can be defined as 'unwanted sound.'",
    "Noise can inhibit learning and complicate health issues, as confirmed by scientific studies.",
    "Noise can cause auditory damage, including tinnitus and hearing loss.",
    "Attempts to regulate noise levels face challenges in defining what constitutes excessive noise.",
    "Emergency warnings like foghorns and sirens are categorized as necessary noise.",
    "The Klaxon horn was invented in 1907 and became widely used in traffic.",
    "The perception of noise varies across cultures and languages.",
    "The Industrial Revolution prompted early efforts at noise control.",
    "Julia Barnett Rice founded the Society for the Suppression of Unnecessary Noise in 1906.",
    "Noise can be a form of social control and power.",
    "Silence is often a luxury that only the wealthy can afford.",
    "The noise of urban life is often a struggle for those who cannot escape it.",
    "The decibel scale is logarithmic and accounts for human sensitivity to sound.",
    "In 2022, New York City received nearly fifty thousand noise complaints.",
    "The eruption of Krakatoa in 1883 is considered one of the loudest sounds in modern history.",
    "Noise can be both a source of pleasure and discomfort for individuals.",
    "The concept of stochastic noise has applications in various fields, including economics and decision-making.",
    "Noise studies have evolved to include the impact of technology on sound.",
    "The relationship between noise and music has been explored by various composers and artists.",
    "Yoko Ono is recognized for her contributions to noise music.",
    "Noise can serve as a form of resistance against societal norms and control.",
    "The aesthetics of noise music often blur the lines between noise and music."
] 
 
Claims:
[
    "The article 'What Is Noise?' by Alex Ross explores the multifaceted concept of noise, examining its 
historical, cultural, and scientific dimensions.",
    "Noise has roots in words like 'nuisance' and 'nausea.'",
    "Noise can also be majestic, as seen in religious texts and literature.",
    "The article delves into how different languages and cultures perceive noise, with terms like French 'bruit' 
and German 'Lärm.'",
    "Ross discusses personal experiences with noise, reflecting on how control over noise affects our perception of
it.",
    "Garret Keizer’s book on noise ethics echoes the sentiment that control over noise affects perception.",
    "Ross traces the evolution of noise in music, highlighting avant-garde composers like John Cage and György 
Ligeti.",
    "Avant-garde composers embraced noise as an artistic element.",
    "The article touches on the societal implications of noise, noting how it exposes social divides and serves as 
a tool of power.",
    "Historical efforts to control noise demonstrate the challenges in defining and regulating noise.",
    "The rise of informational noise is traced alongside technological advancements.",
    "Ross discusses how noise intersects with information theory, referencing Claude Shannon’s work on signal 
processing.",
    "The article concludes by reflecting on noise as a form of

======================================================================



Metrics Summary

  - ❌ Summarization (score: 0.3125, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.31 because the summary includes numerous pieces of extra information that are not present in the original text, leading to a significant deviation from the original content. This lack of alignment and the absence of contradictions indicate a poor quality summarization., error: None)
  - ✅ Clarity [GEval] (score: 0.8176781948093353, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear and direct language, effectively summarizing the article's exploration of noise across various dimensions. It avoids excessive jargon, although it references some complex concepts like 'information theory' without explanation. The ideas are generally presented in an accessible manner, but the mention of specific figures and their contributions could be confusing for readers unfamiliar with the subject. Overall, the response is w

✓ Evaluation completed 🎉! (time taken: 23.89s | token cost: 0.00410595 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

------ RESULTS ------
----Metric: Summarization
Score: 0.3125
Passed: False
Reason: The score is 0.31 because the summary includes numerous pieces of extra information that are not present in the original text, leading to a significant deviation from the original content. This lack of alignment and the absence of contradictions indicate a poor quality summarization.
----Metric: Clarity [GEval]
Score: 0.8176781948093353
Passed: True
Reason: The response uses clear and direct language, effectively summarizing the article's exploration of noise across various dimensions. It avoids excessive jargon, although it references some complex concepts like 'information theory' without explanation. The ideas are generally presented in an accessible manner, but the mention of specific figures and their contributions could be confusing for readers unfamiliar with the subject. Overall, the response is well-structured and informative, with minor areas for improvement in clarity.
----Metric: Professiona

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# Enhancements
# 
# New Prompt:


system_prompt="You speak modern, professional English."
#main prompt: include the content of the article and the instructions for the AI to follow to analyze the article.
prompt = f"""
    I will provide you with an article, do the following:
    
    1. Identify the article's title and author.
    2. Provide a statement on what is the relevance of the article to an AI professional in their professional development. Max 1 paragraph.
    3. Write a comprehensive, accurate summary. Be sure NOT to add extra material beyond the original article. Use 10000 input tokens and 1000 output tokens. 
    The article is the following: 
    <article>
    {docs[0].page_content}
    </article>

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Relevance: <relevance>
    Summary: <summary>
"""

response2 = client.responses.parse(
    model="gpt-4o",
    input=[
        { 'role': 'system','content': system_prompt},
        { 'role': 'user','content': prompt,}], 
    text_format= ArticleAnalysis,
    temperature=0.5 
)

display(Markdown(response2.output_text))
summary2 = response2.output_parsed.summary

#Re-evaluate the new response with the same metrics
test_case2 = LLMTestCase(input=input, actual_output=summary2)

result_multi=evaluate(test_cases=[test_case2], metrics=[summary_metric,clarity, professionalism, bias])


print('------ RESULTS ------')
for r in result_multi.test_results:
    for m in r.metrics_data:
        print("Metric:", m.name)
        print("Score:", m.score)
        print("Passed:", m.success)
        print("Reason:", m.reason)

{"title":"What Is Noise?","author":"Alex Ross","relevance":"For an AI professional, this article offers a nuanced exploration of 'noise' that extends beyond acoustics into data and information theory, which are crucial in AI development. Understanding noise in terms of data interference and signal processing is fundamental for designing robust AI systems, particularly in machine learning where distinguishing between noise and signal is critical for model accuracy and efficiency.","summary":"In \"What Is Noise?\" Alex Ross explores the multifaceted concept of noise, tracing its evolution from a mere auditory nuisance to a complex cultural and informational phenomenon. The article examines noise's dual nature as both a negative disturbance and a potential source of artistic inspiration. Historically linked to chaos and disturbance, noise has also been embraced in religious and musical contexts for its sublimity and power. Ross delves into the linguistic variations of the term across different cultures, highlighting how noise has been perceived and interpreted differently. The article also discusses the cultural and historical dimensions of noise, including its role in music and art, and its impact on society. Noise is portrayed as both a tool of power and an indicator of social struggle, particularly in urban environments. The narrative extends to the realm of information theory, where noise is seen as a challenge to effective communication and data transmission. Ross reflects on personal experiences with noise, illustrating the subjective nature of what constitutes noise versus music. The article underscores the ongoing tension between noise and silence, and the societal and technological efforts to manage it. Ultimately, Ross presents noise as an unavoidable aspect of modern life that both disrupts and enriches human experience."  , "input_tokens":10000,"output_tokens":1000}

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Bias [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

...

Truths (limit=None):
[
    "Noise has various meanings, ranging from negative to positive, and can be associated with both chaos and 
beauty.",
    "The word 'noise' is etymologically linked to 'nuisance' and 'nausea'.",
    "Noise can drive people to madness, as illustrated by the character in Poe's 'The Tell-Tale Heart'.",
    "Noise can also be described as joyful, as seen in religious texts like the Psalms and the Book of Ezekiel.",
    "The term 'noise' has been applied beyond acoustics to describe any ambient activity that hinders a signal.",
    "Different languages have distinct terms for noise, with French using 'bruit' and German using 'Lärm' and 
'Geräusch'.",
    "Noise has inspired a vast body of literature and cultural history, including works on noise music and 
noise-based literary criticism.",
    "Samuel Johnson stated that music is the least disagreeable of all noises.",
    "Personal experiences with noise can vary greatly, with some individuals being more sensitive to it than 
others.",
    "Noise can be perceived as an act of aggression when it is imposed on someone without their consent.",
    "Disputes over noise often reveal social tensions and divisions.",
    "The perception of hip-hop as 'Black Noise' reflects a history of sonic dehumanization directed at minority 
groups.",
    "Silence is often a luxury that only the wealthy can afford, while noise is an indicator of struggle for the 
rest of society.",
    "Noise can inhibit learning and complicate health issues, as confirmed by scientific studies.",
    "Attempts to regulate noise levels face challenges in defining what constitutes excessive noise.",
    "Emergency warnings like sirens and foghorns are categorized as necessary noise.",
    "The perception of noise has evolved with technological advancements, leading to increased levels of noise in 
urban environments.",
    "The concept of stochastic noise has been applied in various fields, including economics and political 
polling.",
    "Noise has been a subject of artistic exploration, with composers like Luigi Russolo advocating for the 
appreciation of noise in music.",
    "The relationship between noise and music is complex, with noise sometimes being seen as a form of liberation 
from traditional musical constraints."
] 
 
Claims:
[
    "Alex Ross explores the multifaceted concept of noise in the article 'What Is Noise?'.",
    "The article traces the evolution of noise from a mere auditory nuisance to a complex cultural and 
informational phenomenon.",
    "Noise has a dual nature as both a negative disturbance and a potential source of artistic inspiration.",
    "Historically, noise has been linked to chaos and disturbance.",
    "Noise has been embraced in religious and musical contexts for its sublimity and power.",
    "The article highlights linguistic variations of the term 'noise' across different cultures.",
    "Noise has been perceived and interpreted differently in various cultures.",
    "The article discusses the cultural and historical dimensions of noise, including its role in music and art.",
    "Noise has an impact on society.",
    "Noise is portrayed as both a tool of power and an indicator of social struggle, particularly in urban 
environments.",
    "In information theory, noise is seen as a challenge to effective communication and data transmission.",
    "Ross reflects on personal experiences with noise, illustrating the subjective nature of what constitutes noise
versus music.",
    "The article underscores the ongoing tension between noise and silence.",
    "There are societal and technological efforts to manage noise.",
    "Ross presents noise as an unavoidable aspect of modern life that both disrupts and enriches human experience."
] 
 
Assessment Questions:
[
    "Does the author talk about their own experience with noise?",
    "Does noise refer only to an acoustic phenomenon?",
    "Are notifications and algorithmic suggestions considered noise?",
  

======================================================================



Metrics Summary

  - ✅ Summarization (score: 0.75, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.75 because the summary includes extra information not found in the original text, which may lead to confusion or misinterpretation. However, there are no contradictions, and the summary captures the main ideas effectively., error: None)
  - ✅ Clarity [GEval] (score: 0.7982346170461578, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear and direct language, effectively conveying the complex ideas surrounding noise. It avoids jargon and presents the evolution of noise in a way that is easy to follow. However, some parts could be more concise, as the length may lead to slight confusion for readers unfamiliar with the topic., error: None)
  - ✅ Professionalism [GEval] (score: 0.9679178692681616, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response maintains a professional tone throu

✓ Evaluation completed 🎉! (time taken: 24.38s | token cost: 0.00400845 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

------ RESULTS ------
Metric: Summarization
Score: 0.75
Passed: True
Reason: The score is 0.75 because the summary includes extra information not found in the original text, which may lead to confusion or misinterpretation. However, there are no contradictions, and the summary captures the main ideas effectively.
Metric: Clarity [GEval]
Score: 0.7982346170461578
Passed: True
Reason: The response uses clear and direct language, effectively conveying the complex ideas surrounding noise. It avoids jargon and presents the evolution of noise in a way that is easy to follow. However, some parts could be more concise, as the length may lead to slight confusion for readers unfamiliar with the topic.
Metric: Professionalism [GEval]
Score: 0.9679178692681616
Passed: True
Reason: The response maintains a professional tone throughout, reflecting expertise in discussing the complex concept of noise. The language used is formal and appropriate for an academic context, avoiding casual expressions. It

Comments on the improvements:


I chose to summarize the New York Times article "What is noise?" by Alex Ross. This is a challenging task for an LLM because it is a very abstract discussion on the social, historical, technical aspects of 'noise'. It discusses noise as an acoustic phenomenon as well aas disturbances more generally, even in the technical sense of the word (as in stochasticity). To tackle this, I use a larger model (gpt-4o instead of gpt-4o-mini) and I set the temperature to 0.5. This allows for some flexibility in the generation, which is needed due to the abstract nature of the text. 

I evaluate the summaries using the Summarization Metric from deepeval. I use a different, smaller, model (pgt-40-mini) to evaluate the answers. I avoid using the same model as the generation because models prefer their own answers. Also, I opt for the smaller version when evaluating because that is a simpler task than generation. Lastly, I set the temperature to be lower than the generation task (0.1 vs 0.5). I include 5 bespoke yes-no questions. Both summaries manage to answer 4 out of 5 questions correctly. 


The initial prompt was a simple statement: "summarize the article in no more than 1000 tokens". However, it turns out that more complex prompts do not necessarily improve the summarization score. Here are a few of the attempts I made and their scores:
    - Summarize the article accurately and concisely in 700 words. Be sure NOT to add extra material beyond the original article. --score=0.4 
    - Produce a concise, neutral summary preserving the core arguments and key evidence.Do not introduce information not present in the article. Preserve uncertainty language. Do not infer unstated intent. Max 700 words. -- score 0.2

Finally, I used the following prompt, which improved the score to 0.75 (pass).
    Write a comprehensive, accurate summary. Be sure NOT to add extra material beyond the original article. Use 1000 input tokens and 1000 output tokens. 

Here I keep the prompt short, but I add instructions to avoid adding extra material beyond the original article, which was the main reason for the failure of the previous prompts. In addition, I set the token use to 10000 input tokens and 1000 output tokens. This encourages the model to use more resources in the summarization. 


Importantly, I noticed that there are substantial differences between runs of the same evaluation metric on the same response output. For example, without changing any code, the initial prompt gets summarization scores between 0.14 and 0.6. This wide range points to a limitation of this evaluation method. It would be important to test many cases, multiple times in order to be more confident in the results. 

In all, the sumamrization score improved significantly. However, the clarity, tone, bias scores were all high in both the original and improved prompts.




# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
